# CRISPR-Guard: Off-Target Risk Prediction Model Training
This notebook contains the complete pipeline for training **CRISPRGuardNet**, a hybrid 1D-CNN + Transformer Encoder architecture for predicting CRISPR-Cas9 off-target gene editing risks. 

### Pipeline Overview:
1. **Dependencies Installation**
2. **Data Loading & Generation** (GUIDE-seq / CRISPR-Net simulated formats)
3. **DNA Sequence One-Hot Encoding** (20bp gRNA + 20bp Target DNA)
4. **Model Architecture Definition** (PyTorch Conv1D + Transformer Encoder blocks)
5. **Model Training & Evaluation** (BCELoss, Adam Optimizer, ROC-AUC metric)
6. **Model Serialization** (Saving weights to `crispr_guard_model.pt`)

In [ ]:
# Install required libraries in Colab
!pip install torch numpy pandas scikit-learn matplotlib

## 1. Import Dependencies
We import PyTorch for deep learning, scikit-learn for evaluation metrics (ROC-AUC), and numpy/pandas for data manipulation.

In [ ]:
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

# Set random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. One-Hot Encoding for gRNA-DNA Pairs
We encode the guide RNA (20bp) and DNA target (20bp) into one-hot vectors. Concatenating them along the channel dimension yields a stacked tensor matrix of shape `(8, 20)`.

In [ ]:
NUCLEOTIDE_INDEX = {"A": 0, "C": 1, "G": 2, "T": 3}

def sequence_to_matrix(sequence: str) -> np.ndarray:
    sequence = sequence.upper().strip()
    L = len(sequence)
    matrix = np.zeros((4, L), dtype=np.float32)
    for i, nuc in enumerate(sequence):
        if nuc in NUCLEOTIDE_INDEX:
            matrix[NUCLEOTIDE_INDEX[nuc], i] = 1.0
    return matrix

def encode_sequence(grna: str, target: str) -> np.ndarray:
    grna_matrix = sequence_to_matrix(grna)    # Shape: (4, 20)
    target_matrix = sequence_to_matrix(target) # Shape: (4, 20)
    combined = np.concatenate([grna_matrix, target_matrix], axis=0) # Shape: (8, 20)
    return combined

## 3. Dataset Generation (GUIDE-seq / CRISPR-Net Simulation)
We simulate a benchmark dataset consisting of active guides, true targets (positives), and highly similar mismatch off-targets (negatives).

In [ ]:
def generate_dummy_data(num_samples=1000):
    nucleotides = ['A', 'C', 'G', 'T']
    data = []
    
    # Generate active gRNAs
    grnas = ["".join(random.choices(nucleotides, k=20)) for _ in range(num_samples // 10)]
    
    for i in range(num_samples):
        grna = random.choice(grnas)
        # Choose label: 1 (positive cut), 0 (negative/no-cut off-target)
        label = random.choice([0, 1])
        
        if label == 1:
            # On-target or 1 mismatch (high cut probability)
            num_mismatches = random.choice([0, 1])
        else:
            # Off-target with 2 to 4 mismatches
            num_mismatches = random.choice([2, 3, 4])
            
        target = list(grna)
        # Randomly mutate positions
        mutate_positions = random.sample(range(20), num_mismatches)
        for pos in mutate_positions:
            original = target[pos]
            target[pos] = random.choice([n for n in nucleotides if n != original])
            
        target_seq = "".join(target)
        data.append({"grna": grna, "target": target_seq, "label": label})
        
    return pd.DataFrame(data)

# Create data
df = generate_dummy_data(2000)
print(df.head())
print(df['label'].value_counts())

## 4. PyTorch Dataset and DataLoader
We wrap the data inside a PyTorch custom dataset for batched training.

In [ ]:
class CRISPRDataset(Dataset):
    def __init__(self, dataframe):
        self.grnas = dataframe['grna'].values
        self.targets = dataframe['target'].values
        self.labels = dataframe['label'].values
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        x = encode_sequence(self.grnas[idx], self.targets[idx])
        y = np.array([self.labels[idx]], dtype=np.float32)
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

train_df, val_df = train_test_split(df, test_size=0.2, random_state=SEED)

train_dataset = CRISPRDataset(train_df)
val_dataset = CRISPRDataset(val_df)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

## 5. CRISPRGuardNet Model Architecture
The network is a hybrid model containing:
- **1D-CNN layers**: Extracts regional features / local motifs from nucleotides.
- **Transformer Encoder**: Captures global dependency across all positions (PAM-seed region modeling).
- **Fully Connected Head**: Outputs cleavage probability via Sigmoid activation.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 20, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, : x.size(1), :]
        return self.dropout(x)

class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, dropout: float = 0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, padding=kernel_size // 2),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout)
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)

class CRISPRGuardNet(nn.Module):
    def __init__(self,
                 in_channels: int = 8,
                 cnn_channels_1: int = 64,
                 cnn_channels_2: int = 128,
                 d_model: int = 128,
                 nhead: int = 2,
                 num_encoder_layers: int = 2,
                 dim_feedforward: int = 256,
                 dropout: float = 0.2,
                 seq_len: int = 20):
        super().__init__()
        self.conv1 = ConvBlock(in_channels, cnn_channels_1, kernel_size=3, dropout=dropout)
        self.conv2 = ConvBlock(cnn_channels_1, cnn_channels_2, kernel_size=3, dropout=dropout)
        self.positional_encoding = PositionalEncoding(d_model=d_model, max_len=seq_len, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.conv2(x)
        x = x.permute(0, 2, 1)
        x = self.positional_encoding(x)
        x = self.transformer_encoder(x)
        x = x.permute(0, 2, 1)
        x = self.global_avg_pool(x).squeeze(-1)
        return self.classifier(x)

## 6. Training Loop & Validation
We train the model using BCELoss and the Adam optimizer. We track validation ROC-AUC to ensure high-fidelity classification.

In [ ]:
model = CRISPRGuardNet().to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 10
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(x_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x_batch.size(0)
        
    train_loss /= len(train_loader.dataset)
    
    # Validation
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            preds = model(x_batch)
            loss = criterion(preds, y_batch)
            val_loss += loss.item() * x_batch.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(y_batch.cpu().numpy())
            
    val_loss /= len(val_loader.dataset)
    
    # Calculate ROC-AUC
    fpr, tpr, _ = roc_curve(all_targets, all_preds)
    val_auc = auc(fpr, tpr)
    
    print(f"Epoch {epoch}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val ROC-AUC: {val_auc:.4f}")

## 7. Metrics Visualization
Plot the ROC curve and Precision-Recall curve to visualize validation set performance.

In [ ]:
# Get final validation predictions
model.eval()
all_preds = []
all_targets = []
with torch.no_grad():
    for x_batch, y_batch in val_loader:
        x_batch = x_batch.to(device)
        preds = model(x_batch)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y_batch.cpu().numpy())

fpr, tpr, _ = roc_curve(all_targets, all_preds)
roc_auc = auc(fpr, tpr)

precision, recall, _ = precision_recall_curve(all_targets, all_preds)
pr_auc = average_precision_score(all_targets, all_preds)

plt.figure(figsize=(12, 5))

# ROC Curve
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, color='forestgreen', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")

# PR Curve
plt.subplot(1, 2, 2)
plt.plot(recall, precision, color='darkorange', lw=2, label=f'PR curve (AUC = {pr_auc:.4f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="lower left")

plt.tight_layout()
plt.show()

## 8. Export Model Weights
Save the model state dictionary. These weights can be placed directly in the backend repository root: `backend/crispr_guard_model.pt`.

In [ ]:
model_path = "crispr_guard_model.pt"
torch.save(model.state_dict(), model_path)
print(f"Successfully exported model state dictionary to {model_path}")